In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np


url = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"
columns = [
    "checking_status", "duration", "credit_history", "purpose", "credit_amount",
    "savings_status", "employment", "installment_commitment", "personal_status",
    "other_parties", "residence_since", "property_magnitude", "age",
    "other_payment_plans", "housing", "existing_credits", "job",
    "num_dependents", "own_telephone", "foreign_worker", "class"
]

df = pd.read_csv(url, sep=" ", header=None, names=columns)
print(df.shape)


(1000, 21)


In [2]:
df['bad_risk'] = df['class'].map({1: 0, 2: 1})

df['monthly_repayment'] = df['credit_amount'] / df['duration']
df['log_credit_amount'] = np.log(df['credit_amount'])
df['age_band'] = pd.cut(df['age'], bins=[18, 25, 40, 60, 100],
                        labels=['young', 'adult', 'middle_aged', 'senior'])

In [3]:
from sklearn.model_selection import train_test_split
y = df['bad_risk']
X = df.drop(columns=['class', 'bad_risk'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

(800, 23) (200, 23)
0.3 0.3


In [4]:

X_train.columns.tolist()

['checking_status',
 'duration',
 'credit_history',
 'purpose',
 'credit_amount',
 'savings_status',
 'employment',
 'installment_commitment',
 'personal_status',
 'other_parties',
 'residence_since',
 'property_magnitude',
 'age',
 'other_payment_plans',
 'housing',
 'existing_credits',
 'job',
 'num_dependents',
 'own_telephone',
 'foreign_worker',
 'monthly_repayment',
 'log_credit_amount',
 'age_band']

In [5]:
import numpy as np
from sklearn.model_selection import RepeatedStratifiedKFold
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
print(cv.get_n_splits())

15


In [6]:
train = X_train.copy()
train['bad_risk'] = y_train
train.groupby('checking_status')['bad_risk'].mean()

,bad_risk
checking_status,
A11,0.479452
A12,0.392523
A13,0.254902
A14,0.120253


In [7]:
train.groupby('savings_status')['bad_risk'].agg(['mean', 'count']).sort_values('mean')

,mean,count
savings_status,,
A64,0.121951,41
A63,0.160000,50
A65,0.166667,144
A62,0.333333,90
A61,0.364211,475


In [8]:
train.groupby('employment')['bad_risk'].agg(['mean', 'count']).sort_values('mean')

,mean,count
employment,,
A74,0.223022,139
A75,0.262136,206
A73,0.276753,271
A71,0.414634,41
A72,0.440559,143


In [9]:
train.groupby('credit_history')['bad_risk'].agg(['mean', 'count']).sort_values('mean')
#moving to one-hot .

,mean,count
credit_history,,
A34,0.175000,240
A33,0.319444,72
A32,0.322115,416
A31,0.536585,41
A30,0.612903,31


In [10]:
train.groupby('job')['bad_risk'].agg(['mean', 'count']).sort_values('mean')
# moving to one-hot encode.

,mean,count
job,,
A173,0.289628,511
A172,0.291139,158
A174,0.347826,115
A171,0.375000,16


In [11]:
ordinal_cols = ['checking_status', 'savings_status', 'employment', 'age_band']
onehot_cols  = ['purpose', 'personal_status', 'other_parties', 'property_magnitude', 'other_payment_plans', 'housing', 'own_telephone', 'foreign_worker','credit_history','job']   # 10 columns, fill in from the table above
numeric_cols = [c for c in X_train.columns if c not in ordinal_cols + onehot_cols]

print(len(ordinal_cols), len(onehot_cols), len(numeric_cols))
print(numeric_cols)

4 10 9
['duration', 'credit_amount', 'installment_commitment', 'residence_since', 'age', 'existing_credits', 'num_dependents', 'monthly_repayment', 'log_credit_amount']


In [12]:
checking_order   = ['A14', 'A13', 'A12', 'A11']
savings_order    = ['A64', 'A63', 'A65', 'A62', 'A61']
employment_order = ['A75', 'A74', 'A73', 'A72', 'A71']
age_order        = ['senior', 'middle_aged', 'adult', 'young']

In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ('ord', OrdinalEncoder(
            categories=[checking_order, savings_order, employment_order, age_order],
            handle_unknown='use_encoded_value', unknown_value=-1),
         ordinal_cols),
        ('ohe', OneHotEncoder(handle_unknown='ignore'), onehot_cols),
    ],
    remainder='passthrough'   # numeric columns pass through unchanged
)

In [14]:
X_check = preprocessor.fit_transform(X_train)
print(X_check.shape)

(800, 53)


In [15]:
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_val_score

pipe = Pipeline([
    ('prep', preprocessor),
    ('model', DummyClassifier(strategy='most_frequent'))
])

acc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='accuracy')
auc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc')

print(len(acc), acc.mean(), acc.std())
print(auc.mean())

15 0.6999999999999998 1.1102230246251565e-16
0.5
